# Course 5: Building a Small Bangla Language Model
## Session 6: Training on free GPUs

**Goal:** train with a schedule, mixed precision and resumable checkpoints, and read the loss curves.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every code cell here was run once on a CPU (November 2026) using the small built-in sample. Cells marked TODO or NOT RUN need your data, a GPU or the network. The run here is a tiny CPU demonstration; your real runs need a GPU.

In [ ]:
SAMPLE_LINES = [
    "আমি বাংলায় গান গাই।",
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "বাংলা আমার মাতৃভাষা।",
    "আজ আকাশ খুব পরিষ্কার।",
    "ছোট মেয়েটি নদীর ধারে খেলছিল।",
    "বাবা বাজার থেকে মাছ কিনে আনলেন।",
    "মা রান্নাঘরে ভাত রান্না করছেন।",
    "আমরা সবাই মিলে স্কুলে যাই।",
    "বৃষ্টি হলে মাঠে পানি জমে।",
    "পাখিরা সকালে গান গায়।",
    "দাদু প্রতিদিন সন্ধ্যায় গল্প বলেন।",
    "ছেলেটি বই পড়তে খুব ভালোবাসে।",
    "নদীতে অনেক নৌকা চলছে।",
    "শীতকালে আমরা পিঠা খাই।",
    "ঢাকা বাংলাদেশের রাজধানী।",
    "আমার বন্ধু ভালো ফুটবল খেলে।",
]

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

def train_bpe(lines, vocab_size, specials=("[UNK]", "<|endoftext|>")):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFC()
    tok.pre_tokenizer = pre_tokenizers.Metaspace()
    tok.decoder = decoders.Metaspace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=list(specials))
    tok.train_from_iterator(lines, trainer)
    return tok

### 1. Encode once, save token IDs to disk

In [ ]:
import numpy as np
tok = train_bpe(SAMPLE_LINES * 20, 300)          # TODO: Tokenizer.from_file("tokenizer.json")
EOT = tok.token_to_id("<|endoftext|>")

def encode_lines(lines, path):
    ids = []
    for line in lines:
        ids += tok.encode(line).ids + [EOT]
    np.array(ids, dtype=np.uint16).tofile(path)
    return len(ids)

print(encode_lines(SAMPLE_LINES[:12] * 40, "train.bin"), "train tokens")   # TODO: lines of train.txt
print(encode_lines(SAMPLE_LINES[12:] * 10, "val.bin"), "val tokens")       # TODO: lines of val.txt

### 2. Batches from a memory-mapped file

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = Config(vocab_size=tok.get_vocab_size(), block_size=32, n_layer=2, n_head=2, d_model=64)   # TODO: your shape

def get_batch(path, batch_size=32):
    data = np.memmap(path, dtype=np.uint16, mode="r")
    ix = np.random.randint(0, len(data) - cfg.block_size - 1, batch_size)
    x = torch.from_numpy(np.stack([data[i:i + cfg.block_size].astype(np.int64) for i in ix]))
    y = torch.from_numpy(np.stack([data[i + 1:i + cfg.block_size + 1].astype(np.int64) for i in ix]))
    return x.to(device), y.to(device)

### 3. Schedule, evaluation and checkpoints

In [ ]:
import os, time
MAX_LR, WARMUP, TOTAL, MIN_LR = 3e-3, 20, 300, 3e-4       # TODO: e.g. 3e-4, 200, your total, 3e-5

def lr_at(step):
    if step < WARMUP:
        return MAX_LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / (TOTAL - WARMUP)
    return MIN_LR + 0.5 * (MAX_LR - MIN_LR) * (1 + math.cos(math.pi * progress))

@torch.no_grad()
def evaluate(model, path, batches=20):
    model.eval()
    losses = [model(*get_batch(path))[1].item() for _ in range(batches)]
    model.train()
    return sum(losses) / len(losses)

def save_ckpt(path, model, opt, step):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "step": step, "cfg": cfg.__dict__}, path)

def load_ckpt(path, model, opt):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    return ck["step"]

### 4. The training loop, resumable

In [ ]:
def train(until_step, ckpt="ckpt.pt"):
    torch.manual_seed(0); np.random.seed(0)
    model = GPT(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=0.1)
    scaler = torch.amp.GradScaler(enabled=(device == "cuda"))
    step = load_ckpt(ckpt, model, opt) if os.path.exists(ckpt) else 0
    log, t0, seen = [], time.time(), 0
    while step < until_step:
        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        x, y = get_batch("train.bin")
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            _, loss = model(x, y)
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        step += 1; seen += x.numel()
        if step % 50 == 0:
            log.append((step, loss.item(), evaluate(model, "val.bin")))
            save_ckpt(ckpt, model, opt, step)
    print(f"reached step {step}, {seen / (time.time() - t0):,.0f} tokens/s")
    return model, log

if os.path.exists("ckpt.pt"):
    os.remove("ckpt.pt")
model, log1 = train(150)        # first "session"
model, log2 = train(300)        # resumes from the checkpoint
for row in log1 + log2:
    print("step %4d  train %.3f  val %.3f" % row)

### 5. Plot the curves

In [ ]:
import matplotlib.pyplot as plt
steps, tr, va = zip(*(log1 + log2))
plt.plot(steps, tr, label="train"); plt.plot(steps, va, label="validation")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.savefig("loss.png")
print("saved loss.png")

### TODO
- Copy `ckpt.pt` somewhere that outlives the session (a Kaggle dataset or the Hugging Face Hub).
- Run a learning-rate search with three short runs.
- Train both shapes from session 5 for the same number of steps and compare validation loss.

On the tiny sample the validation loss stops improving quickly, because the validation sentences never appear in training. That is the overfitting pattern from segment 3.